# Perfilamiento de la Fuente Primaria: ECV 2025 (DANE)
**Proyecto:** ETL Segunda Entrega — Huella de Carbono en Hogares Colombianos (ODS 12)  
**Responsable:** David Aponte (Persona 2 — API y Calidad de Datos)  
**Autores:** Nikol Hurtado, David Aponte, Diana Panesso, Sebastián Celorio  

## Objetivo
Documentar el perfilamiento **sistemático y reproducible** de los 4 capítulos de la ECV 2025,
vinculando cada hallazgo con la regla GX correspondiente (DQ01–DQ11) y su justificación de umbral.
Los umbrales marcados **PROVISIONAL** en `docs/quality_rules.md` se concretan aquí con base empírica.
> **Instrucciones de ejecución:** coloca los CSV de la ECV 2025 en `data/raw/ecv/` antes de correr las celdas.
> Descarga: https://microdatos.dane.gov.co/index.php/catalog/905


## 0. Configuración del entorno

In [ ]:
import os, sys, warnings, re, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 40)

PROJECT_ROOT = Path('..').resolve()
RAW_ECV  = PROJECT_ROOT / 'data' / 'raw' / 'ecv'
REF_DIR  = PROJECT_ROOT / 'data' / 'reference'

ECV_SEP = ';'
ECV_ENC = 'latin-1'
HH_KEYS = ['DIRECTORIO', 'SECUENCIA_P']

ECV_TABLES = {
    'servicios':       {'keywords': ['servicios'],               'columns': [*HH_KEYS, 'P5018', 'P5010', 'P8536', 'P8540', 'P3163']},
    'vivienda':        {'keywords': ['vivienda'],                'columns': [*HH_KEYS, 'P1_DEPARTAMENTO', 'P1_MUNICIPIO']},
    'caracteristicas': {'keywords': ['caracter'],                'columns': [*HH_KEYS, 'ORDEN']},
    'condiciones':     {'keywords': ['condiciones', 'tenencia'], 'exclude': ['programa'],
                        'columns': [*HH_KEYS, 'P1077S15', 'P1077S11', 'P1077S12']},
}

dane_ref     = pd.read_csv(REF_DIR / 'dane_departamentos.csv', sep=';')
CODIGOS_DANE = set(dane_ref['codigo_dane'].astype(int))

def _norm(text):
    text = unicodedata.normalize('NFKD', str(text))
    return ''.join(c for c in text if not unicodedata.combining(c)).lower()

def find_ecv(spec, folder=RAW_ECV):
    for p in sorted(folder.glob('*.csv')):
        n = _norm(p.name)
        if all(k in n for k in spec['keywords']) and not any(x in n for x in spec.get('exclude', [])):
            return p
    return None

def load_chapter(table):
    spec = ECV_TABLES[table]
    path = find_ecv(spec)
    if path is None:
        print(f'[AVISO] CSV del capítulo {table!r} no encontrado en {RAW_ECV}.')
        print('        Descarga la ECV 2025 y coloca los CSV en data/raw/ecv/')
        return None, None
    hdr = pd.read_csv(path, sep=ECV_SEP, encoding=ECV_ENC, nrows=0)
    by_name = {str(c).strip().upper(): c for c in hdr.columns}
    missing = [c for c in spec['columns'] if c not in by_name]
    if missing:
        raise ValueError(f'[{table}] Columnas faltantes: {missing}')
    df = pd.read_csv(path, sep=ECV_SEP, encoding=ECV_ENC,
                     usecols=[by_name[c] for c in spec['columns']],
                     keep_default_na=False, na_values=[''])
    df.columns = [str(c).strip().upper() for c in df.columns]
    return df[spec['columns']], path

print(f'Raíz: {PROJECT_ROOT}')
print(f'Departamentos DANE ({len(CODIGOS_DANE)}): {sorted(CODIGOS_DANE)}')


---
## 1. Capítulo «Servicios del Hogar»
Variables: `P5018` (gasto electricidad COP/mes), `P5010` (estrato), `P8536/P8540/P3163` (cocción).


In [ ]:
df_serv, path_serv = load_chapter('servicios')
if df_serv is not None:
    print(f'Archivo: {path_serv.name}')
    print(f'Filas: {len(df_serv):,}  Columnas: {df_serv.shape[1]}')
    display(df_serv.head())


### 1.1 Grano y Duplicados — DQ02 (Critical) y DQ03 (Warning ≥ 99 %)

- **DQ02 (Critical 100 %):** `DIRECTORIO` o `SECUENCIA_P` nulos → hogar imposible de cruzar con ninguna dimensión → bloqueo inmediato.
- **DQ03 (Warning mostly=0.99):** La fase 1 registró 212 duplicados en 88,432 filas (0.24 %). El umbral de 99 % deja margen para variaciones menores mientras alerta ante problemas de carga (>1 %).


In [ ]:
if df_serv is not None:
    n_total  = len(df_serv)
    n_dir    = df_serv['DIRECTORIO'].isna().sum()
    n_seq    = df_serv['SECUENCIA_P'].isna().sum()
    n_uniq   = df_serv.drop_duplicates(subset=HH_KEYS).shape[0]
    n_dup    = n_total - n_uniq
    pct_uniq = n_uniq / n_total * 100

    res = pd.DataFrame({
        'Regla':      ['DQ02 — DIRECTORIO nulo', 'DQ02 — SECUENCIA_P nulo', 'DQ03 — Duplicados HH_KEY'],
        'Valor':      [n_dir, n_seq, n_dup],
        'Pct':        [f'{n_dir/n_total*100:.4f}%', f'{n_seq/n_total*100:.4f}%', f'{n_dup/n_total*100:.4f}%'],
        'Umbral':     ['0%', '0%', '≤1%'],
        'Severidad':  ['Critical', 'Critical', 'Warning'],
        'Estado':     [
            'PASA' if n_dir == 0 else 'FALLA',
            'PASA' if n_seq == 0 else 'FALLA',
            'PASA' if pct_uniq >= 99 else 'FALLA',
        ],
    })
    display(res)
    print(f'Unicidad observada: {pct_uniq:.4f}%')
    print(f'Justificación DQ03: fase 1 eliminó {n_dup} duplicados ({n_dup/n_total*100:.2f}%). Umbral=99% cubre el patrón histórico con margen.')
else:
    print('Sin datos reales. Fase 1: 212 duplicados en 88,432 filas (0.24%). Umbral DQ03 mostly=0.99 justificado.')


### 1.2 Gasto de Electricidad `P5018` — DQ04 (Warning ≥ 70 %)

- **Causa de nulos (~23 %):** hogares con servicios incluidos en el arriendo no reportan el valor del recibo.
- **Umbral 70 %:** con <70 % de datos observados, la huella eléctrica dependería más de la imputación que del dato real, haciendo el KPI poco confiable para R1 y R6.
- **Decisión de pipeline:** imputación por mediana de estrato (determinista); ceros de estratos 2–6 se reemplazan por la mediana del estrato.
- **Outliers:** valores >P₉₉ corresponden a cobros comerciales o errores de digitación → truncar.


In [ ]:
if df_serv is not None:
    df_serv['P5018'] = pd.to_numeric(df_serv['P5018'], errors='coerce')
    n_total = len(df_serv)
    n_obs   = df_serv['P5018'].notna().sum()
    pct_obs = n_obs / n_total * 100
    obs = df_serv['P5018'].dropna()
    p99 = obs.quantile(0.99)
    n_out = int((obs > p99).sum())

    print('P5018 — Gasto mensual electricidad (COP/mes)')
    print(f'  Observados:  {n_obs:,}  ({pct_obs:.2f}%)')
    print(f'  Nulos:       {n_total - n_obs:,}  ({100 - pct_obs:.2f}%)')
    print(f'  DQ04 ≥70%: {"PASA" if pct_obs >= 70 else "FALLA"}')
    print()
    desc = obs.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99])
    print(desc.to_string())
    print(f'\n  P99 = {p99:,.0f} COP/mes  → umbral de truncamiento')
    print(f'  Outliers (>P99): {n_out:,}  ({n_out/n_total*100:.3f}%)  → cobros comerciales o error de digitación')
else:
    print('Sin datos reales. Fase 1: ~77.4% completitud. Umbral DQ04 ≥70% justificado.')
    print('P99 ≈ 350,000 COP/mes. Outliers = cobros no residenciales o errores de captura.')


### 1.3 Estrato `P5010` — DQ05 (Warning ≥ 99.9 %)

Colombia define 6 estratos residenciales (1–6). Códigos 7 y 8 corresponden a propiedades no estratificadas (<0.02 % en F1).  
**Umbral 99.9 %:** base en la frecuencia histórica observada (12 casos en 88,432 filas).  
**Decisión:** reclasificar a 0 ("No válido/No informa"). Tarifa fallback = estrato 1.


In [ ]:
if df_serv is not None:
    df_serv['P5010'] = pd.to_numeric(df_serv['P5010'], errors='coerce')
    dist_e = df_serv['P5010'].value_counts(dropna=False).sort_index()
    n_inv   = int((~df_serv['P5010'].between(1, 6)).sum())
    pct_val = df_serv['P5010'].between(1, 6).mean() * 100

    print('Distribución de P5010 (estrato):')
    print(dist_e.to_string())
    print(f'\n  Fuera de [1,6]: {n_inv}  ({100 - pct_val:.4f}%)')
    print(f'  DQ05 ≥99.9%:   {"PASA" if pct_val >= 99.9 else "FALLA"}')
    print('  Decisión: reclasificar 7/8 → 0; tarifa fallback = estrato 1 (más bajo).')
else:
    print('Sin datos reales. Fase 1: 12 de 88,432 con código 7/8 (0.0136%). DQ05 ≥99.9% justificado.')


---
## 2. Capítulo «Datos de la Vivienda» — DQ06, DQ07, DQ08 (todos Critical)

`P1_DEPARTAMENTO` es la **llave de integración** entre la ECV y la API IDEAM.  
- **DQ07 (Critical 100%):** Nulo en la llave → hogar no puede recibir clima → R6/R7 inviables para ese hogar.  
- **DQ08 (Critical 100%):** Código fuera del catálogo DIVIPOLA → JOIN incorrecto (clima de otro dpto) → KPIs contaminados. Base contractual: catálogo oficial DANE.


In [ ]:
df_viv, path_viv = load_chapter('vivienda')
if df_viv is not None:
    print(f'Archivo: {path_viv.name}  Filas: {len(df_viv):,}')
    df_viv['P1_DEPARTAMENTO'] = pd.to_numeric(df_viv['P1_DEPARTAMENTO'], errors='coerce')
    n_null  = int(df_viv['P1_DEPARTAMENTO'].isna().sum())
    en_cat  = df_viv['P1_DEPARTAMENTO'].dropna().astype(int).isin(CODIGOS_DANE)
    n_inv   = int((~en_cat).sum())
    pct_comp = (1 - n_null / len(df_viv)) * 100
    pct_val  = en_cat.mean() * 100

    print(f'\nDQ07 — P1_DEPARTAMENTO nulos: {n_null}  (completitud {pct_comp:.2f}%) → {"PASA" if n_null == 0 else "FALLA"}')
    print(f'DQ08 — Fuera catálogo:       {n_inv}   ({100 - pct_val:.4f}%)       → {"PASA" if n_inv == 0 else "FALLA"}')

    dist = (df_viv['P1_DEPARTAMENTO'].value_counts()
            .rename_axis('codigo_dane').reset_index(name='hogares')
            .assign(codigo_dane=lambda d: d['codigo_dane'].astype('Int64'))
            .merge(dane_ref, on='codigo_dane', how='left')
            .sort_values('codigo_dane'))
    print(f'\nDepartamentos distintos en ECV: {df_viv["P1_DEPARTAMENTO"].dropna().nunique()}')
    display(dist)
else:
    print('Sin datos reales. P1_DEPARTAMENTO: 100% completo esperado (campo obligatorio DANE).')
    print('DQ08: todos los códigos dentro del catálogo DIVIPOLA (33 departamentos).')


---
## 3. Capítulo «Características y Composición» — DQ09 (Critical 100 %)

Este capítulo tiene **una fila por persona**. Se usa para contar integrantes por hogar
(group-count por `DIRECTORIO + SECUENCIA_P` → `dim_hogar.num_personas`).  
**DQ09:** Sin llaves válidas el group-count es imposible → columna nula en `dim_hogar` → R5 roto.


In [ ]:
df_car, path_car = load_chapter('caracteristicas')
if df_car is not None:
    print(f'Archivo: {path_car.name}  Filas: {len(df_car):,}  (personas, no hogares)')
    n_dir = int(df_car['DIRECTORIO'].isna().sum())
    n_seq = int(df_car['SECUENCIA_P'].isna().sum())
    print(f'DQ09 — DIRECTORIO nulos:   {n_dir} → {"PASA" if n_dir == 0 else "FALLA"}')
    print(f'DQ09 — SECUENCIA_P nulos:  {n_seq} → {"PASA" if n_seq == 0 else "FALLA"}')

    personas_hh = df_car.groupby(HH_KEYS).size()
    print('\nPersonas por hogar — estadísticas:')
    print(personas_hh.describe().to_string())
else:
    print('Sin datos reales. DQ09: DIRECTORIO y SECUENCIA_P sin nulos esperado.')
    print('Promedio personas/hogar Colombia ≈ 3.0 según DANE.')


---
## 4. Capítulo «Condiciones de Vida» — DQ10 (Critical) y DQ11 (Warning)

Variables clave: `P1077S15` (tenencia carro — R3), `P1077S11` y `P1077S12` (equipos de climatización — R7).  
El DANE define dominio `{1: Sí, 2: No}` para todas las variables de tenencia de bienes. Un valor fuera de `{1, 2}` es un error de codificación.


In [ ]:
df_cond, path_cond = load_chapter('condiciones')
if df_cond is not None:
    print(f'Archivo: {path_cond.name}  Filas: {len(df_cond):,}')
    for var, desc, regla, sev in [
        ('P1077S15', 'Carro particular (R3)', 'DQ10', 'Critical'),
        ('P1077S11', 'Aire acondicionado (R7)', 'DQ11', 'Warning'),
        ('P1077S12', 'Ventilador o abanico (R7)', 'DQ11', 'Warning'),
    ]:
        df_cond[var] = pd.to_numeric(df_cond[var], errors='coerce')
        dist  = df_cond[var].value_counts(dropna=False).sort_index()
        n_inv = int((~df_cond[var].isin([1, 2])).sum())
        pct_v = df_cond[var].isin([1, 2]).mean() * 100
        estado = 'PASA' if n_inv == 0 else 'FALLA'
        print(f'\n{var} — {desc}  [{regla}, {sev}]')
        print(dist.to_string())
        print(f'  Fuera de {{1,2}}: {n_inv}  ({100 - pct_v:.4f}%) → {estado}')
else:
    print('Sin datos reales. Fase 1: P1077S15/S11/S12 en dominio {1,2}. Sin valores fuera de rango.')


### Reformulación de R3 (evidencia del perfilamiento)

> La ECV **no captura variables de movilidad** (km/año, consumo de galones, cilindraje).  
> `P1077S15` es puramente binario: posesión de carro (Sí/No).  
> **Decisión de ingeniería (documentada aquí):** R3 reformulado como
> *«comparar la huella residencial (eléctrica + cocción) entre hogares con y sin automóvil particular,
> controlando por estrato socioeconómico»*.


---
## 5. Verificación de Contrato Columnar (DQ01, DQ06, DQ09_col, DQ10_col)

Cada suite GX arranca con `ExpectTableColumnsToMatchSet` (Critical). Si falta una columna requerida,
el pipeline no puede continuar (la información no se puede fabricar).


In [ ]:
rows_c = []
for tabla, spec in ECV_TABLES.items():
    path = find_ecv(spec)
    if path is None:
        rows_c.append({'Tabla': tabla, 'Encontrado': False,
                        'Col_req': len(spec['columns']), 'Col_pres': 'N/A', 'Missing': 'N/A'})
        continue
    hdr = pd.read_csv(path, sep=ECV_SEP, encoding=ECV_ENC, nrows=0)
    cols_up = [str(c).strip().upper() for c in hdr.columns]
    miss = [c for c in spec['columns'] if c not in cols_up]
    rows_c.append({
        'Tabla': tabla, 'Encontrado': True,
        'Col_req': len(spec['columns']), 'Col_pres': len(cols_up),
        'Missing': miss if miss else 'Ninguna',
    })
display(pd.DataFrame(rows_c))


---
## 6. Matriz de Síntesis — Hallazgos y Decisiones del Pipeline


In [ ]:
sintesis = pd.DataFrame([
    {'Capítulo':'Servicios',      'Atributo':'DIRECTORIO+SECUENCIA_P', 'Dim':'Completitud',       'Regla':'DQ02','Sev':'Critical','Umbral':'100%',    'F1_obs':'~100%',    'Base_umbral':'Grano de hechos — sin llaves la fila no existe'},
    {'Capítulo':'Servicios',      'Atributo':'DIRECTORIO+SECUENCIA_P', 'Dim':'Unicidad',           'Regla':'DQ03','Sev':'Warning', 'Umbral':'≥99%',     'F1_obs':'99.76%',   'Base_umbral':'F1: 212 dups en 88,432 filas (0.24%); >1% = error de carga'},
    {'Capítulo':'Servicios',      'Atributo':'P5018',                  'Dim':'Completitud',       'Regla':'DQ04','Sev':'Warning', 'Umbral':'≥70%',     'F1_obs':'~77%',     'Base_umbral':'<70% → imputación domina el KPI eléctrico (R1,R6)'},
    {'Capítulo':'Servicios',      'Atributo':'P5010',                  'Dim':'Validez',           'Regla':'DQ05','Sev':'Warning', 'Umbral':'≥99.9%',   'F1_obs':'~99.99%',  'Base_umbral':'F1: 12 hogares cod 7/8 (0.014%); reclasificar a 0'},
    {'Capítulo':'Vivienda',       'Atributo':'columnas req.',           'Dim':'Validez (contrato)','Regla':'DQ06','Sev':'Critical','Umbral':'100%',    'F1_obs':'100%',     'Base_umbral':'Contrato fuente — columna faltante = pipeline roto'},
    {'Capítulo':'Vivienda',       'Atributo':'P1_DEPARTAMENTO',        'Dim':'Completitud',       'Regla':'DQ07','Sev':'Critical','Umbral':'100%',    'F1_obs':'~100%',    'Base_umbral':'Llave de integración con API IDEAM (R6,R7)'},
    {'Capítulo':'Vivienda',       'Atributo':'P1_DEPARTAMENTO',        'Dim':'Validez',           'Regla':'DQ08','Sev':'Critical','Umbral':'100%',    'F1_obs':'100%',     'Base_umbral':'Catálogo DIVIPOLA 33 depts — código inválido = JOIN incorrecto'},
    {'Capítulo':'Características','Atributo':'DIRECTORIO+SECUENCIA_P', 'Dim':'Completitud',       'Regla':'DQ09','Sev':'Critical','Umbral':'100%',    'F1_obs':'~100%',    'Base_umbral':'Sin llaves, group-count integrantes imposible (R5)'},
    {'Capítulo':'Condiciones',    'Atributo':'P1077S15',               'Dim':'Validez',           'Regla':'DQ10','Sev':'Critical','Umbral':'100%',    'F1_obs':'100%',     'Base_umbral':'Diccionario DANE: dominio contractual {1:Sí, 2:No}'},
    {'Capítulo':'Condiciones',    'Atributo':'P1077S11, P1077S12',     'Dim':'Validez',           'Regla':'DQ11','Sev':'Warning', 'Umbral':'100%',    'F1_obs':'~100%',    'Base_umbral':'Warning: alimenta solo R7 (S11 aire acondicionado, S12 ventilador; dict. DANE ECV 2025)'},
])
display(sintesis)
